# 06 -- Design Catalog

Explore the trade-off space between aerodynamic performance and manufacturability:

1. **Build catalog** — baseline, v2 optimized, and alpha-blended designs
2. **Aero evaluation** — 10-target surrogate (with CG correction + control derivatives)
3. **Manufacturability scoring** — curvature, thickness, mold complexity
4. **Pareto analysis** — L/D vs. manufacturability trade-off front
5. **Geometric comparison** — planform overlays, radar charts
6. **STEP export** — batch export for downstream structural/manufacturing pipeline

In [ ]:
import sys
sys.path.insert(0, '..')

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from src.config import load_all
from src.optimization.catalog import DesignCatalog
from src.evaluation.manufacturability import compute_manufacturability
from src.visualization.comparison import (
    plot_pareto, plot_radar, plot_planform_overlay, plot_summary_table,
)

from src.visualization.style import apply_style, COLORS
apply_style()

cfg = load_all()
mission = cfg['mission']
feasibility = cfg['feasibility']

## 1. Build the Design Catalog

Two reference points + 4 interpolated designs spanning the full trade-off spectrum.

In [ ]:
catalog = DesignCatalog()

# ── Reference designs ──
catalog.add_baseline()

# Load v2 optimized design
try:
    catalog.add_optimized('../output/best_x_v2.npy')
    print('Using v2 optimized design')
except FileNotFoundError:
    catalog.add_optimized('../output/best_x.npy')
    print('Using v1 design (fallback)')

# ── Top feasible designs from the LHS dataset ──
from src.optimization.database import EvaluationDatabase

db = EvaluationDatabase.load('../data/eval_database.json')
X_arr, results = db.to_arrays()

# Find top 5 feasible designs by L/D (diverse)
feasible_idx = [i for i, r in enumerate(results) 
                if r.get('is_feasible', False) and r.get('L_over_D', 0) > 0]
feasible_idx.sort(key=lambda i: -results[i]['L_over_D'])

# Add top 5 with diversity filter (min distance between designs)
from src.parameterization.design_variables import get_bounds_arrays
lb, ub = get_bounds_arrays()
diag = np.linalg.norm(ub - lb)
selected = []
for idx in feasible_idx:
    x = X_arr[idx]
    # Skip if too close to an already selected design
    if any(np.linalg.norm(x - s) / diag < 0.05 for s in selected):
        continue
    selected.append(x)
    name = f'lhs_top{len(selected)}'
    catalog.add(name, x, origin='dataset', tags=['lhs'])
    if len(selected) >= 5:
        break

print(f'Added {len(selected)} top feasible designs from LHS dataset')

# ── Alpha-blends between best LHS and optimized ──
if len(selected) > 0:
    best_lhs = f'lhs_top1'
    catalog.interpolate(best_lhs, 'optimized', [0.33, 0.67])

print(f'\nCatalog: {len(catalog)} designs')
for e in catalog:
    p = e.params
    print(f'  {e.name:20s}  span={2*p.half_span:.2f}m  t/c={p.body_tc_root:.3f}  '
          f'taper={p.taper_ratio:.3f}  sweep={p.le_sweep_deg:.1f}')

## 2. Aerodynamic Evaluation (Surrogate)

Instant evaluation via the pre-trained MLP ensemble (7 VLM primitives → full reconstruction).

In [ ]:
catalog.evaluate_aero(
    mission=mission,
    feasibility=feasibility,
    use_surrogate=False,  # AVL validation — surrogat has 30% L/D gap
)

## 3. Manufacturability Scoring

Geometric metrics quantifying fabrication difficulty: twist/dihedral gradients, minimum thickness, mold complexity, taper severity.

In [ ]:
catalog.evaluate_manufacturability()

# Detailed breakdown for baseline vs optimized
for name in ['baseline', 'optimized']:
    e = catalog[name]
    mm = e.manufacturing_metrics
    print(f'\n=== {name.upper()} (score={mm["manufacturability_score"]:.3f}) ===')
    for k, v in mm.items():
        if k.startswith('sub_'):
            print(f'  {k[4:]:25s}  {v:.3f}')
    print(f'  {"twist_gradient_max":25s}  {mm["twist_gradient_max"]:.1f} °/m')
    print(f'  {"dihedral_gradient_max":25s}  {mm["dihedral_gradient_max"]:.1f} °/m')
    print(f'  {"thickness_tip_mm":25s}  {mm["thickness_tip_mm"]:.1f} mm')
    print(f'  {"n_dihedral_breaks":25s}  {mm["n_dihedral_breaks"]}')

## 4. Summary Table

In [ ]:
from src.geometry.control_surfaces import compute_control_surface_geometry

print(catalog.summary())
print()

# Pandas DataFrame for detailed comparison
df = catalog.to_dataframe()

# Add control surface dimension columns
elevon_spans = []
aileron_spans = []
for e in catalog:
    cs_list = compute_control_surface_geometry(e.params)
    ev_span = 0.0
    ai_span = 0.0
    for cs in cs_list:
        if 'elevon' in cs.name.lower():
            ev_span = cs.span * 100  # m -> cm
        elif 'aileron' in cs.name.lower():
            ai_span = cs.span * 100  # m -> cm
    elevon_spans.append(ev_span)
    aileron_spans.append(ai_span)

df['elevon_span_cm'] = elevon_spans
df['aileron_span_cm'] = aileron_spans

cols = ['origin', 'L_over_D', 'static_margin', 'is_feasible',
        'manuf_manufacturability_score', 'struct_mass', 'endurance_min',
        'half_span', 'body_tc_root', 'taper_ratio',
        'elevon_deflection', 'Cl_beta', 'x_cg_frac',
        'elevon_span_cm', 'aileron_span_cm']
display(df[[c for c in cols if c in df.columns]].round(3))

## 5. Pareto Plot — L/D vs. Manufacturability

The fundamental trade-off: aerodynamic performance vs. fabrication ease. The Pareto front shows the best achievable compromise at each performance level.

In [ ]:
fig = plot_pareto(catalog, save_path='../output/catalog_pareto.png')

## 6. Radar Chart — Multi-Criteria Comparison

Normalized comparison across 6 dimensions: L/D, stability, manufacturability, volume, endurance, and structural mass.

In [ ]:
# Compare key designs: baseline, top LHS designs, blends, and optimized
radar_designs = [n for n in catalog.names if n != 'baseline'][:6]  # skip infeasible baseline
fig = plot_radar(catalog, designs=radar_designs,
                 save_path='../output/catalog_radar.png')
plt.show()

## 7. Geometric Comparison — Planform Overlays

Top view (planform) and front view (dihedral) superimposed for all catalog designs.

In [ ]:
fig = plot_planform_overlay(catalog, save_path='../output/catalog_planforms.png')
plt.show()

## 8. Summary Table (visual)

In [ ]:
fig = plot_summary_table(catalog, save_path='../output/catalog_table.png')
plt.show()

In [ ]:
# 9. Batch STEP Export
# Select which designs to export (uncomment to run — takes ~30s per design)
# paths = catalog.export_all_step(
#     output_dir='../output/catalog',
#     version='v2',
#     n_profile=100,
# )
# for name, path in paths.items():
#     print(f'  {name}: {path}')

In [ ]:
# 10. Save Catalog
catalog.save('../output/catalog.json')
print(f'Catalog saved: {len(catalog)} designs → output/catalog.json')

# To reload in another session:
# catalog = DesignCatalog.load('../output/catalog.json')